In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import sys
from sklearn.metrics import mean_squared_error

import warnings 
warnings.filterwarnings('ignore') 

sys.path.append("../../")
import stan


## Loading ST dataset
We use a Visium spatial transcriptomics dataset of the human glioblastoma, which is publicly available from the 10x genomics website. For first time use, the dataset will be downloaded to the directory data/Parent_Visium_Human_Glioblastoma by default.

In [2]:
adata = sc.datasets.visium_sge(sample_id="Parent_Visium_Human_Glioblastoma")
adata.var_names_make_unique()
adata

/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 3468 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

We perform some basic filtering of genes and spots.

In [3]:
sc.pp.filter_genes(adata, min_cells=5)
sc.pp.filter_cells(adata, min_counts=500)
adata

AnnData object with n_obs × n_vars = 3462 × 20950
    obs: 'in_tissue', 'array_row', 'array_col', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells'
    uns: 'spatial'
    obsm: 'spatial'

In [4]:
!mkdir results_glioblastoma/
adata.write("results_glioblastoma/glioblastoma.h5ad")

mkdir: results_brain/: File exists


## 02 filter

In [ ]:
adata = sc.read_h5ad("results_brain/01_brain.h5ad")
adata.layers['raw'] = adata.X
adata

In [ ]:
adata.to_df().to_csv('inputs_count/brain.csv')

In [ ]:
adata.obs['ncounts'] = adata.to_df('raw').T.sum()
sc.pl.spatial(adata, color='ncounts', size=1.5, alpha_img=0)
adata.obs['ncounts'].hist(bins=100)

In [ ]:
adata = stan.add_gene_tf_matrix(adata,
                                min_cells_proportion = 0.2,
                                min_tfs_per_gene= 5,
                                min_genes_per_tf= 10,
                                gene_tf_source="hTFtarget",
                                tf_list="humantfs",
                                source_dir="data/gene_tf/")

D = adata.varm['gene_tf']
Y = adata.to_df()
print('gene-cell matrix: {} x {}'.format(Y.shape[1], Y.shape[0]))
print('gene-TF matrix: {} x {}'.format(D.shape[0], D.shape[1]))
print('min cells per gene: {}'.format((Y>0).sum().min()))
print('min genes per cell: {}'.format((Y>0).T.sum().min()))
print('min tfs per gene: {}'.format(D.T.abs().sum().min()))
print('min genes per tf: {}'.format(D.abs().sum().min()))

In [ ]:
stan.pixel_intensity(adata, windowsize=25)
stan.make_kernel(adata, n=250, im_feats_weight=0.05, bandwidth=0.2)

In [ ]:
sc.pp.normalize_total(adata)
adata.layers['scaled'] = np.sqrt(adata.to_df())

In [ ]:
adata.write("results_brain/02_brain.h5ad")

## 03 Ridge

In [ ]:
adata = sc.read_h5ad("results_brain/02_brain.h5ad")
random.seed(0)
stan.assign_folds(adata, n_folds=10)

In [ ]:
ridge_model = stan.Ridge(adata, layer='scaled')
ridge_model.fit(n_steps=5, stages=1,
                grid_search_params={'lam':[1e-4, 1e4]})

In [ ]:
cor, gene_cor = ridge_model.evaluate(fold=-1)
adata.obs['pred_cor_ridge'] = cor
adata.var['pred_cor_ridge'] = gene_cor

adata.obsm['tfa_ridge'] = pd.DataFrame(ridge_model.W_concat.T, index=adata.obs_names, columns=adata.uns['tf_names'])

In [ ]:
print(ridge_model.params)
print("Sample Cor:" + str(round(np.nanmedian(cor), 4)))
print(" Gene Cor: " + str(round(np.nanmedian(gene_cor), 4)))

In [ ]:
adata.write("results_brain/03_brain.h5ad")

In [ ]:
# check
Y = adata.varm['gene_tf'].dot(ridge_model.W_concat)
mean_squared_error(Y, adata.to_df('scaled').T)

## 04 STAN

In [ ]:
adata = sc.read_h5ad("results_brain/03_brain.h5ad")
stan_model = stan.Stan(adata, layer='scaled')
stan_model.fit(n_steps=5, stages=1,
              grid_search_params={'lam2':[1e-4, 1e4],'lam1':[1e-4, 1e4]})

In [ ]:
cor, gene_cor = stan_model.evaluate(fold=-1)
adata.obs['pred_cor_stan'] = cor
adata.var['pred_cor_stan'] = gene_cor

adata.obsm['tfa_stan'] = pd.DataFrame(stan_model.W_concat.T, index=adata.obs_names, columns=adata.uns['tf_names'])

In [ ]:
print(stan_model.params)
print("Sample Cor:" + str(round(np.nanmedian(cor), 4)))
print(" Gene Cor: " + str(round(np.nanmedian(gene_cor), 4)))

In [ ]:
xstring = "pred_cor_ridge"
ystring = "pred_cor_stan"
adata.obs['n_counts_10000'] = adata.obs['n_counts'] / 10000
lim_min = np.minimum(adata.obs[xstring], adata.obs[ystring])
lim_max = np.maximum(adata.obs[xstring], adata.obs[ystring])
plt.plot([lim_min, lim_max], [lim_min, lim_max], '-', alpha=0.25, color='grey')
sns.scatterplot(data=adata.obs, x=xstring, y=ystring, s=10, hue="n_counts_10000", linewidth=0, palette='flare')
plt.ylabel("STAN")
plt.xlabel("Ridge")
plt.legend(title="UMI\nCount\n(1e4)", loc='upper right', bbox_to_anchor=(1.3, 1), columnspacing=0.5, ncol=1, handletextpad=0, frameon=False)
plt.title('Cross Validation Performance\n(Pearson Correlation)')

In [ ]:
adata_tfa = AnnData(
    X = adata.obsm['tfa_stan'],
    obs = adata.obs,
    obsm = {name: obj for (name, obj) in adata.obsm.items() if "tf" not in name},
    layers = {name: obj for (name, obj) in adata.obsm.items() if "tf" in name})
adata_tfa.uns = adata.uns

In [ ]:
adata.write("results_brain/04_brain.h5ad")
adata_tfa.write("results_brain/04_brain_tfa.h5ad")

In [ ]:
# check
Y = adata.varm['gene_tf'].dot(stan_model.W_concat)
mean_squared_error(Y, adata.to_df('scaled').T)